## Sentiment Analysis (Semantic Matching)

This notebook mirrors `src/sentiment_analysis.py`, but makes it explicit which **resume** each set of rankings belongs to, and adds a TF-IDF approach that combines:

- **Skills dictionary score** (skills overlap/coverage)
- **TF-IDF cosine similarity** between resume text and job text

Then it runs the same composite-scoring pipeline with embeddings:

- **MiniLM** (`sentence-transformers/all-MiniLM-L6-v2`)
- **MPNet** (`sentence-transformers/all-mpnet-base-v2`)


In [3]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Add ../src to import path (so we can import modules like the other notebooks)
src_path = (Path.cwd().parent / 'src').resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from sentiment_analysis import ResumeJobMatcher
from skills import SkillVocabularyBuilder, DEFAULT_SYNONYM_MAP
from matching import dictionary_compare

# Override load_processed_data to use files from /content/
class PatchedResumeJobMatcher(ResumeJobMatcher):
    @staticmethod
    def load_processed_data():
        data_path = Path("/content")
        jobs_path = data_path / "jobs_subset.csv"
        resumes_path = data_path / "resumes_subset.csv"

        if not jobs_path.exists():
            raise FileNotFoundError(f"Jobs file not found: {jobs_path}")
        if not resumes_path.exists():
            raise FileNotFoundError(f"Resumes file not found: {resumes_path}")

        jobs_df = pd.read_csv(jobs_path)
        resumes_df = pd.read_csv(resumes_path)
        return jobs_df, resumes_df

ResumeJobMatcher = PatchedResumeJobMatcher

TOP_K = 5
# How many resumes to run. Set to None to run ALL resumes (can be slow).
MAX_RESUMES = 10
RESUME_START_INDEX = 0

# Weights for the TF-IDF + skills_dictionary hybrid score
TFIDF_WEIGHT = 0.4
DICT_WEIGHT = 0.6

# Prefer the fastest available accelerator
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'[INFO] Torch device: {DEVICE}')

jobs_df, resumes_df = ResumeJobMatcher.load_processed_data()
if MAX_RESUMES is None:
    RESUME_INDICES = list(range(RESUME_START_INDEX, len(resumes_df)))
else:
    RESUME_INDICES = list(range(
        RESUME_START_INDEX,
        min(RESUME_START_INDEX + int(MAX_RESUMES), len(resumes_df)),
    ))

print(f'[INFO] Ranking for {len(RESUME_INDICES)} resumes (of {len(resumes_df)})')

# Build vocabulary from real job skills (same as src/sentiment_analysis.py)
vocab = ResumeJobMatcher.build_skill_vocabulary_from_jobs(
    jobs_df,
    skills_col='job_skills_cleaned',
    min_frequency=3,
)
print(f'[INFO] Vocabulary size: {len(vocab)}')

# Jobs as records (description/skills/title) for the composite scoring ranker
jobs = ResumeJobMatcher.prepare_jobs_as_records(jobs_df)

# For resume context, use the original processed resume columns
RESUME_ID_COL = 'ID'
RESUME_TEXT_COL = 'Resume_str'
RESUME_CLEAN_COL = 'Resume_cleaned'
RESUME_CATEGORY_COL = 'Category'

if RESUME_TEXT_COL not in resumes_df.columns:
    raise KeyError(f'Expected {RESUME_TEXT_COL} in resumes_df columns: {resumes_df.columns.tolist()}')

builder = SkillVocabularyBuilder(synonym_map=DEFAULT_SYNONYM_MAP)
vocab_set = set(vocab.keys()) | set(builder.synonym_map.values())

def extract_skills_from_text(text: str, max_ngram: int = 4) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []

    cleaned = ResumeJobMatcher.clean_text(text)
    tokens = re.findall(r'[a-zA-Z0-9][a-zA-Z0-9+#./\-]*', cleaned)

    found: list[str] = []
    for n in range(1, max_ngram + 1):
        for i in range(len(tokens) - n + 1):
            phrase = ' '.join(tokens[i : i + n])
            phrase_n = builder.normalize_skill(phrase)
            if not builder.is_valid_skill_phrase(phrase_n):
                continue
            canonical = builder.canonicalize(phrase_n)
            if canonical in vocab_set:
                found.append(canonical)

    # Also parse delimiter-like lists (if present)
    for item in builder.parse_skills(text):
        item_n = builder.normalize_skill(item)
        if not builder.is_valid_skill_phrase(item_n):
            continue
        canonical = builder.canonicalize(item_n)
        if canonical in vocab_set:
            found.append(canonical)

    # De-dup preserve order
    return list(dict.fromkeys(found))

def resume_context_df(indices: list[int]) -> pd.DataFrame:
    rows = []
    for idx in indices:
        r = resumes_df.iloc[int(idx)].fillna('')
        resume_id = r.get(RESUME_ID_COL, idx)
        category = r.get(RESUME_CATEGORY_COL, '')
        text = str(r.get(RESUME_TEXT_COL, ''))
        cleaned = str(r.get(RESUME_CLEAN_COL, ''))
        skills = extract_skills_from_text(text)
        rows.append({
            'resume_index': int(idx),
            'resume_id': str(resume_id),
            'category': str(category),
            'resume_preview': (cleaned[:220] + '…') if len(cleaned) > 220 else cleaned,
            'extracted_skills_preview': ', '.join(skills[:20]),
            'num_extracted_skills': int(len(skills)),
        })
    return pd.DataFrame(rows)

# Precompute job texts/skills once for speed
job_clean_texts = [ResumeJobMatcher.clean_text(str(x)) for x in jobs_df.fillna('')['job_combined_text'].tolist()]
job_skills_raw = jobs_df.fillna('')['job_skills_cleaned'].astype(str).tolist()

# Cache for sentence-transformers models + precomputed job embeddings
ST_MODELS: dict[str, object] = {}
JOB_EMBEDDINGS: dict[str, np.ndarray] = {}

def get_st_model(model_name: str):
    if model_name in ST_MODELS:
        return ST_MODELS[model_name]
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name, device=DEVICE)
    ST_MODELS[model_name] = model
    return model

def get_job_embeddings(model_name: str, batch_size: int = 128) -> np.ndarray:
    if model_name in JOB_EMBEDDINGS:
        return JOB_EMBEDDINGS[model_name]
    model = get_st_model(model_name)
    embs = model.encode(
        job_clean_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=batch_size,
        show_progress_bar=True,
    )
    JOB_EMBEDDINGS[model_name] = embs
    return embs

resume_context_df(RESUME_INDICES)


[INFO] Torch device: cuda
[INFO] Ranking for 10 resumes (of 712)
[INFO] Vocabulary size: 2600


,resume_index,resume_id,category,resume_preview,extracted_skills_preview,num_extracted_skills
0,0,20345168,ACCOUNTANT,staff accountant summary skilled accountant su...,"deadlines, accounting, software, payroll, mana...",67
1,1,18569929,ACCOUNTANT,payroll accountant summary sixteen years exper...,"payroll, accounting, quickbooks, excel, word, ...",42
2,2,12338274,ACCOUNTANT,accountant summary to pursue excellence in the...,"excellence, business, placement, honesty, orga...",49
3,3,25067742,ACCOUNTANT,accountant summary if you need someone who del...,"accounting, responsibility, entertainment, tra...",27
4,4,78403342,ACCOUNTANT,accountant summary self-motivated accountant o...,"auditing, finance, communication, management, ...",60
5,5,13072019,ACCOUNTANT,project accountant career focus dedicated and ...,"responsibility, coordination, business, writin...",57
6,6,39674178,ACCOUNTANT,"staff accountant executive summary motivated, ...","analytical, accounting, training, judgment, bu...",38
7,7,31602598,ACCOUNTANT,accountant summary capable accountant successf...,"deadlines, accounting, software, technology, m...",57
8,8,80053367,ACCOUNTANT,general accountant career focus to obtain a po...,"certification, physical, assessment, care, nur...",64
9,9,27637576,ACCOUNTANT,corporate accountant summary strategic and ana...,"analytical, finance, analysis, budget, forecas...",76


### Resume context

This table shows the resume metadata/features used for ranking outputs below (so it’s clear which resume each ranking belongs to).

In [4]:
resume_context_df(RESUME_INDICES)


,resume_index,resume_id,category,resume_preview,extracted_skills_preview,num_extracted_skills
0,0,20345168,ACCOUNTANT,staff accountant summary skilled accountant su...,"deadlines, accounting, software, payroll, mana...",67
1,1,18569929,ACCOUNTANT,payroll accountant summary sixteen years exper...,"payroll, accounting, quickbooks, excel, word, ...",42
2,2,12338274,ACCOUNTANT,accountant summary to pursue excellence in the...,"excellence, business, placement, honesty, orga...",49
3,3,25067742,ACCOUNTANT,accountant summary if you need someone who del...,"accounting, responsibility, entertainment, tra...",27
4,4,78403342,ACCOUNTANT,accountant summary self-motivated accountant o...,"auditing, finance, communication, management, ...",60
5,5,13072019,ACCOUNTANT,project accountant career focus dedicated and ...,"responsibility, coordination, business, writin...",57
6,6,39674178,ACCOUNTANT,"staff accountant executive summary motivated, ...","analytical, accounting, training, judgment, bu...",38
7,7,31602598,ACCOUNTANT,accountant summary capable accountant successf...,"deadlines, accounting, software, technology, m...",57
8,8,80053367,ACCOUNTANT,general accountant career focus to obtain a po...,"certification, physical, assessment, care, nur...",64
9,9,27637576,ACCOUNTANT,corporate accountant summary strategic and ana...,"analytical, finance, analysis, budget, forecas...",76


### 1) Skills dictionary + TF-IDF (hybrid similarity)

Computes, for each resume-job pair:
- `dictionary_score` from `src/matching.py` (skill coverage + Jaccard)
- `tfidf_similarity` cosine similarity between resume text and job text

Then combines them into `hybrid_score = DICT_WEIGHT * dict_norm + TFIDF_WEIGHT * tfidf_similarity`.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

all_rows = []
for resume_index in RESUME_INDICES:
    r = resumes_df.iloc[int(resume_index)].fillna('')
    resume_id = str(r.get(RESUME_ID_COL, resume_index))
    category = str(r.get(RESUME_CATEGORY_COL, ''))
    resume_text = str(r.get(RESUME_TEXT_COL, ''))

    resume_skills = extract_skills_from_text(resume_text)
    resume_clean = ResumeJobMatcher.clean_text(resume_text)

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
    matrix = vectorizer.fit_transform([resume_clean] + job_clean_texts)
    tfidf_scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()

    for job_index, job_rec in enumerate(jobs):
        dict_result = dictionary_compare(resume_skills, job_skills_raw[job_index])
        dict_score = float(dict_result['dictionary_score'])
        dict_norm = dict_score / 100.0
        tfidf_sim = float(tfidf_scores[job_index])
        hybrid = (DICT_WEIGHT * dict_norm) + (TFIDF_WEIGHT * tfidf_sim)

        all_rows.append({
            'resume_index': int(resume_index),
            'resume_id': resume_id,
            'category': category,
            'job_index': int(job_index),
            'title': job_rec.get('title', ''),
            'company': job_rec.get('company', ''),
            'dictionary_score': dict_score,
            'tfidf_similarity': tfidf_sim,
            'hybrid_score': float(hybrid),
            'matched_skills_preview': ', '.join(dict_result.get('matched_skills', [])[:10]),
            'missing_skills_preview': ', '.join(dict_result.get('missing_skills', [])[:10]),
        })

hybrid_df = pd.DataFrame(all_rows)
hybrid_df = hybrid_df.sort_values(['resume_index', 'hybrid_score'], ascending=[True, False])

hybrid_topk = (
    hybrid_df
    .groupby(['resume_index', 'resume_id', 'category'], as_index=False, sort=False)
    .head(TOP_K)
)
hybrid_topk['rank'] = hybrid_topk.groupby(['resume_index']).cumcount() + 1
hybrid_topk[[
    'resume_index','resume_id','category','rank',
    'job_index','title','company',
    'hybrid_score','dictionary_score','tfidf_similarity',
    'matched_skills_preview','missing_skills_preview',
]]


<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
/tmp/ipykernel_712/3581405975.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hybrid_topk['rank'] = hybrid_topk.groupby(['resume_index']).cumcount() + 1


,resume_index,resume_id,category,rank,job_index,title,company,hybrid_score,dictionary_score,tfidf_similarity,matched_skills_preview,missing_skills_preview
629,0,20345168,ACCOUNTANT,1,629,"Sales Consultant - Rehoboth, Delaware",Winebow,0.202030,31.19,0.037225,"excel, marketing, outlook, sales, word","core commission, diver, driver s license, ios ..."
783,0,20345168,ACCOUNTANT,2,783,Human Resources Generalist,Creative Foam Products LLC,0.197134,27.45,0.081086,"accounting, compensation, hiring, human resour...","ada, bachelor s degree, benefits, business adm..."
1373,0,20345168,ACCOUNTANT,3,1373,Sr. Accountant-Track to-Assistant Controller (...,Robert Half,0.192811,25.61,0.097878,"accounting, construction, engineering, excel, ...","accruals, architecture, budgeting, cost accoun..."
2380,0,20345168,ACCOUNTANT,4,2380,Billing Insurance Coordinator //Pay rate: $25/hr,Stellar Professionals,0.192406,29.64,0.036414,"accounting, business, excel, microsoft office","claims processing, finance, insurance plans, m..."
830,0,20345168,ACCOUNTANT,5,830,Front Desk Manager,Crunch Fitness,0.192371,31.19,0.013077,"customer service, hiring, management, marketin...","communication skills, computer skills, fitness..."
3629,1,18569929,ACCOUNTANT,1,629,"Sales Consultant - Rehoboth, Delaware",Winebow,0.205146,32.23,0.029414,"excel, marketing, outlook, sales, word","core commission, diver, driver s license, ios ..."
3958,1,18569929,ACCOUNTANT,2,958,Accounts Receivable Accountant,Motion Recruitment,0.172201,20.89,0.117151,"accounting, accounts receivable, collections","cash flow, customer reconciliations, financial..."
4373,1,18569929,ACCOUNTANT,3,1373,Sr. Accountant-Track to-Assistant Controller (...,Robert Half,0.169521,22.17,0.091252,"accounting, accruals, excel, financial stateme...","architecture, budgeting, construction, cost ac..."
5269,1,18569929,ACCOUNTANT,4,2269,Finance Manager,Blue Grass Manufacturing,0.164416,19.25,0.122290,"accounting, accounts payable, accounts receiva...",7+ years of experience in finance and accounti...
4188,1,18569929,ACCOUNTANT,5,1188,Property Accounting Supervisor,LHH,0.153904,20.89,0.071410,"accounting, excel, general ledger","bank reconciliation, bs in accounting, compute..."


### 2) MiniLM (same composite scoring logic as `src/sentiment_analysis.py`)

Uses `ResumeJobMatcher.rank_jobs(...)` with `sentence-transformers/all-MiniLM-L6-v2` embeddings (and TF-IDF fallback if the model can't be loaded).

In [6]:
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

# Encode all jobs once (on DEVICE) and reuse across resumes
job_embs = get_job_embeddings(MODEL_NAME, batch_size=128)

# Use ResumeJobMatcher weights/heuristics for the composite score, but reuse semantic scores from embeddings
matcher = ResumeJobMatcher(vocabulary=vocab.keys(), model_name=MODEL_NAME)
print(f'[INFO] Using {MODEL_NAME} on device={DEVICE} (backend=sentence-transformers if available)')

def keyword_set(text: str, top_k: int = 25) -> set[str]:
    kws = matcher.extracts_keywords(text, top_k=top_k)
    return set(kws)

def required_labels(job_clean: str) -> set[str]:
    required = set()
    for label, patterns in matcher.qual_patterns.items():
        if any(re.search(p, job_clean) for p in patterns):
            required.add(label)
    return required

# Precompute job-side keyword sets + required labels once
JOB_KEYWORDS = [keyword_set(t, top_k=25) for t in job_clean_texts]
JOB_REQUIRED = [required_labels(t) for t in job_clean_texts]

rows = []
model = get_st_model(MODEL_NAME)

for resume_index in RESUME_INDICES:
    r = resumes_df.iloc[int(resume_index)].fillna('')
    resume_id = str(r.get(RESUME_ID_COL, resume_index))
    category = str(r.get(RESUME_CATEGORY_COL, ''))
    resume_text = str(r.get(RESUME_TEXT_COL, ''))
    resume_clean = ResumeJobMatcher.clean_text(resume_text)

    # Resume-side features once per resume
    resume_skills = matcher.extract_skills(resume_text)
    resume_keywords = keyword_set(resume_clean, top_k=25)

    resume_has_label = {}
    for label, patterns in matcher.qual_patterns.items():
        resume_has_label[label] = any(re.search(p, resume_clean) for p in patterns)

    # Semantic similarity via dot product (embeddings are normalized)
    resume_emb = model.encode([resume_clean], convert_to_numpy=True, normalize_embeddings=True)[0]
    semantic_scores = (job_embs @ resume_emb).ravel()

    # Candidate pool to keep scoring fast
    CANDIDATE_POOL = 300
    cand_idx = np.argsort(semantic_scores)[::-1][:CANDIDATE_POOL]

    scored = []
    for j in cand_idx:
        job_clean = job_clean_texts[int(j)]

        # Skills overlap (use the same extract_skills logic, but extract skills for this job on-demand)
        job_skills = matcher.extract_skills(job_clean)
        overlap_score = matcher.jaccard_similarity(resume_skills, job_skills)

        job_keywords = JOB_KEYWORDS[int(j)]
        keyword_score = (len(resume_keywords & job_keywords) / len(job_keywords)) if job_keywords else 0.0

        req = JOB_REQUIRED[int(j)]
        if not req:
            qualification_score = 1.0
        else:
            missing = [lab for lab in req if not resume_has_label.get(lab, False)]
            qualification_score = 1.0 - (len(missing) / len(req))

        semantic = float(semantic_scores[int(j)])
        overall = (
            matcher.weights['semantic_similarity'] * semantic
            + matcher.weights['skill_overlap'] * overlap_score
            + matcher.weights['keyword_relevance'] * keyword_score
            + matcher.weights['qualification_match'] * qualification_score
        )
        scored.append((overall, int(j), semantic, overlap_score, keyword_score, qualification_score))

    scored.sort(key=lambda t: t[0], reverse=True)
    top = scored[:TOP_K]

    for rank, (overall, job_index, semantic, overlap, keyword, qual) in enumerate(top, start=1):
        rows.append({
            'resume_index': int(resume_index),
            'resume_id': resume_id,
            'category': category,
            'rank': int(rank),
            'job_index': int(job_index),
            'title': jobs[job_index].get('title',''),
            'company': jobs[job_index].get('company',''),
            'overall_score': round(float(overall) * 100.0, 2),
            'semantic_similarity': round(float(semantic), 4),
            'skill_overlap_score': round(float(overlap), 4),
            'keyword_relevance': round(float(keyword), 4),
            'qualification_match': round(float(qual), 4),
        })

df = pd.DataFrame(rows)
df


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Using sentence-transformers/all-MiniLM-L6-v2 on device=cuda (backend=sentence-transformers if available)


,resume_index,resume_id,category,rank,job_index,title,company,overall_score,semantic_similarity,skill_overlap_score,keyword_relevance,qualification_match
0,0,20345168,ACCOUNTANT,1,698,Sr. Staff Accountant,Robert Half,47.69,0.7349,0.1134,0.20,1.0
1,0,20345168,ACCOUNTANT,2,204,Hybrid Senior Tax Manager - Partnerships/S-Cor...,CyberCoders,46.73,0.6776,0.1261,0.24,1.0
2,0,20345168,ACCOUNTANT,3,958,Accounts Receivable Accountant,Motion Recruitment,46.37,0.7043,0.1233,0.16,1.0
3,0,20345168,ACCOUNTANT,4,280,Staff Accountant,Willamette Falls Paper Company,46.13,0.6663,0.1546,0.16,1.0
4,0,20345168,ACCOUNTANT,5,1203,Senior Accountant - General Ledger,ParetoHealth,45.90,0.6527,0.1101,0.28,1.0
5,1,18569929,ACCOUNTANT,1,2731,Accountant/Staff Accountant,Intellectt Inc,44.69,0.6718,0.0735,0.24,1.0
6,1,18569929,ACCOUNTANT,2,958,Accounts Receivable Accountant,Motion Recruitment,43.79,0.6236,0.0962,0.24,1.0
7,1,18569929,ACCOUNTANT,3,1203,Senior Accountant - General Ledger,ParetoHealth,43.72,0.6269,0.0909,0.24,1.0
8,1,18569929,ACCOUNTANT,4,280,Staff Accountant,Willamette Falls Paper Company,43.27,0.5945,0.1447,0.16,1.0
9,1,18569929,ACCOUNTANT,5,1494,Accounts Receivable Clerk,ASK Consulting,42.17,0.5874,0.1719,0.04,1.0


### 3) MPNet top-k recommendations (same composite scoring logic)

Runs the exact same `ResumeJobMatcher.rank_jobs(...)` flow, but swaps the embedding model to MPNet.

In [7]:
MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'

# Encode all jobs once (on DEVICE) and reuse across resumes
job_embs = get_job_embeddings(MODEL_NAME, batch_size=128)

# Use ResumeJobMatcher weights/heuristics for the composite score, but reuse semantic scores from embeddings
matcher = ResumeJobMatcher(vocabulary=vocab.keys(), model_name=MODEL_NAME)
print(f'[INFO] Using {MODEL_NAME} on device={DEVICE} (backend=sentence-transformers if available)')

def keyword_set(text: str, top_k: int = 25) -> set[str]:
    kws = matcher.extracts_keywords(text, top_k=top_k)
    return set(kws)

def required_labels(job_clean: str) -> set[str]:
    required = set()
    for label, patterns in matcher.qual_patterns.items():
        if any(re.search(p, job_clean) for p in patterns):
            required.add(label)
    return required

# Precompute job-side keyword sets + required labels once
JOB_KEYWORDS = [keyword_set(t, top_k=25) for t in job_clean_texts]
JOB_REQUIRED = [required_labels(t) for t in job_clean_texts]

rows = []
model = get_st_model(MODEL_NAME)

for resume_index in RESUME_INDICES:
    r = resumes_df.iloc[int(resume_index)].fillna('')
    resume_id = str(r.get(RESUME_ID_COL, resume_index))
    category = str(r.get(RESUME_CATEGORY_COL, ''))
    resume_text = str(r.get(RESUME_TEXT_COL, ''))
    resume_clean = ResumeJobMatcher.clean_text(resume_text)

    # Resume-side features once per resume
    resume_skills = matcher.extract_skills(resume_text)
    resume_keywords = keyword_set(resume_clean, top_k=25)

    resume_has_label = {}
    for label, patterns in matcher.qual_patterns.items():
        resume_has_label[label] = any(re.search(p, resume_clean) for p in patterns)

    # Semantic similarity via dot product (embeddings are normalized)
    resume_emb = model.encode([resume_clean], convert_to_numpy=True, normalize_embeddings=True)[0]
    semantic_scores = (job_embs @ resume_emb).ravel()

    # Candidate pool to keep scoring fast
    CANDIDATE_POOL = 300
    cand_idx = np.argsort(semantic_scores)[::-1][:CANDIDATE_POOL]

    scored = []
    for j in cand_idx:
        job_clean = job_clean_texts[int(j)]

        # Skills overlap (use the same extract_skills logic, but extract skills for this job on-demand)
        job_skills = matcher.extract_skills(job_clean)
        overlap_score = matcher.jaccard_similarity(resume_skills, job_skills)

        job_keywords = JOB_KEYWORDS[int(j)]
        keyword_score = (len(resume_keywords & job_keywords) / len(job_keywords)) if job_keywords else 0.0

        req = JOB_REQUIRED[int(j)]
        if not req:
            qualification_score = 1.0
        else:
            missing = [lab for lab in req if not resume_has_label.get(lab, False)]
            qualification_score = 1.0 - (len(missing) / len(req))

        semantic = float(semantic_scores[int(j)])
        overall = (
            matcher.weights['semantic_similarity'] * semantic
            + matcher.weights['skill_overlap'] * overlap_score
            + matcher.weights['keyword_relevance'] * keyword_score
            + matcher.weights['qualification_match'] * qualification_score
        )
        scored.append((overall, int(j), semantic, overlap_score, keyword_score, qualification_score))

    scored.sort(key=lambda t: t[0], reverse=True)
    top = scored[:TOP_K]

    for rank, (overall, job_index, semantic, overlap, keyword, qual) in enumerate(top, start=1):
        rows.append({
            'resume_index': int(resume_index),
            'resume_id': resume_id,
            'category': category,
            'rank': int(rank),
            'job_index': int(job_index),
            'title': jobs[job_index].get('title',''),
            'company': jobs[job_index].get('company',''),
            'overall_score': round(float(overall) * 100.0, 2),
            'semantic_similarity': round(float(semantic), 4),
            'skill_overlap_score': round(float(overlap), 4),
            'keyword_relevance': round(float(keyword), 4),
            'qualification_match': round(float(qual), 4),
        })

df = pd.DataFrame(rows)
df


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Using sentence-transformers/all-mpnet-base-v2 on device=cuda (backend=sentence-transformers if available)


,resume_index,resume_id,category,rank,job_index,title,company,overall_score,semantic_similarity,skill_overlap_score,keyword_relevance,qualification_match
0,0,20345168,ACCOUNTANT,1,280,Staff Accountant,Willamette Falls Paper Company,51.84,0.8293,0.1546,0.16,1.0
1,0,20345168,ACCOUNTANT,2,2937,***Accounting Senior Manager | Hybrid in Tampa...,Vaco,50.48,0.7348,0.1759,0.24,1.0
2,0,20345168,ACCOUNTANT,3,698,Sr. Staff Accountant,Robert Half,50.29,0.8091,0.1134,0.20,1.0
3,0,20345168,ACCOUNTANT,4,1203,Senior Accountant - General Ledger,ParetoHealth,49.96,0.7688,0.1101,0.28,1.0
4,0,20345168,ACCOUNTANT,5,204,Hybrid Senior Tax Manager - Partnerships/S-Cor...,CyberCoders,49.95,0.7697,0.1261,0.24,1.0
5,1,18569929,ACCOUNTANT,1,280,Staff Accountant,Willamette Falls Paper Company,47.83,0.7248,0.1447,0.16,1.0
6,1,18569929,ACCOUNTANT,2,2731,Accountant/Staff Accountant,Intellectt Inc,47.58,0.7545,0.0735,0.24,1.0
7,1,18569929,ACCOUNTANT,3,1203,Senior Accountant - General Ledger,ParetoHealth,47.34,0.7304,0.0909,0.24,1.0
8,1,18569929,ACCOUNTANT,4,958,Accounts Receivable Accountant,Motion Recruitment,44.66,0.6483,0.0962,0.24,1.0
9,1,18569929,ACCOUNTANT,5,496,Payroll Assistant Manager,"Cheney Brothers, Inc.",43.94,0.6941,0.0986,0.08,1.0


### 4) BGE-small top-k recommendations (same composite scoring logic)

Runs the exact same `ResumeJobMatcher.rank_jobs(...)` flow, but swaps the embedding model to BGE-small (`BAAI/bge-small-en-v1.5`).

In [8]:
MODEL_NAME = 'BAAI/bge-small-en-v1.5'

# Encode all jobs once (on DEVICE) and reuse across resumes
job_embs = get_job_embeddings(MODEL_NAME, batch_size=128)

# Use ResumeJobMatcher weights/heuristics for the composite score, but reuse semantic scores from embeddings
matcher = ResumeJobMatcher(vocabulary=vocab.keys(), model_name=MODEL_NAME)
print(f'[INFO] Using {MODEL_NAME} on device={DEVICE} (backend=sentence-transformers if available)')

def keyword_set(text: str, top_k: int = 25) -> set[str]:
    kws = matcher.extracts_keywords(text, top_k=top_k)
    return set(kws)

def required_labels(job_clean: str) -> set[str]:
    required = set()
    for label, patterns in matcher.qual_patterns.items():
        if any(re.search(p, job_clean) for p in patterns):
            required.add(label)
    return required

# Precompute job-side keyword sets + required labels once
JOB_KEYWORDS = [keyword_set(t, top_k=25) for t in job_clean_texts]
JOB_REQUIRED = [required_labels(t) for t in job_clean_texts]

rows = []
model = get_st_model(MODEL_NAME)

for resume_index in RESUME_INDICES:
    r = resumes_df.iloc[int(resume_index)].fillna('')
    resume_id = str(r.get(RESUME_ID_COL, resume_index))
    category = str(r.get(RESUME_CATEGORY_COL, ''))
    resume_text = str(r.get(RESUME_TEXT_COL, ''))
    resume_clean = ResumeJobMatcher.clean_text(resume_text)

    # Resume-side features once per resume
    resume_skills = matcher.extract_skills(resume_text)
    resume_keywords = keyword_set(resume_clean, top_k=25)

    resume_has_label = {}
    for label, patterns in matcher.qual_patterns.items():
        resume_has_label[label] = any(re.search(p, resume_clean) for p in patterns)

    # Semantic similarity via dot product (embeddings are normalized)
    resume_emb = model.encode([resume_clean], convert_to_numpy=True, normalize_embeddings=True)[0]
    semantic_scores = (job_embs @ resume_emb).ravel()

    # Candidate pool to keep scoring fast
    CANDIDATE_POOL = 300
    cand_idx = np.argsort(semantic_scores)[::-1][:CANDIDATE_POOL]

    scored = []
    for j in cand_idx:
        job_clean = job_clean_texts[int(j)]

        # Skills overlap (use the same extract_skills logic, but extract skills for this job on-demand)
        job_skills = matcher.extract_skills(job_clean)
        overlap_score = matcher.jaccard_similarity(resume_skills, job_skills)

        job_keywords = JOB_KEYWORDS[int(j)]
        keyword_score = (len(resume_keywords & job_keywords) / len(job_keywords)) if job_keywords else 0.0

        req = JOB_REQUIRED[int(j)]
        if not req:
            qualification_score = 1.0
        else:
            missing = [lab for lab in req if not resume_has_label.get(lab, False)]
            qualification_score = 1.0 - (len(missing) / len(req))

        semantic = float(semantic_scores[int(j)])
        overall = (
            matcher.weights['semantic_similarity'] * semantic
            + matcher.weights['skill_overlap'] * overlap_score
            + matcher.weights['keyword_relevance'] * keyword_score
            + matcher.weights['qualification_match'] * qualification_score
        )
        scored.append((overall, int(j), semantic, overlap_score, keyword_score, qualification_score))

    scored.sort(key=lambda t: t[0], reverse=True)
    top = scored[:TOP_K]

    for rank, (overall, job_index, semantic, overlap, keyword, qual) in enumerate(top, start=1):
        rows.append({
            'resume_index': int(resume_index),
            'resume_id': resume_id,
            'category': category,
            'rank': int(rank),
            'job_index': int(job_index),
            'title': jobs[job_index].get('title',''),
            'company': jobs[job_index].get('company',''),
            'overall_score': round(float(overall) * 100.0, 2),
            'semantic_similarity': round(float(semantic), 4),
            'skill_overlap_score': round(float(overlap), 4),
            'keyword_relevance': round(float(keyword), 4),
            'qualification_match': round(float(qual), 4),
        })

df_bge = pd.DataFrame(rows)
df_bge

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Using BAAI/bge-small-en-v1.5 on device=cuda (backend=sentence-transformers if available)


,resume_index,resume_id,category,rank,job_index,title,company,overall_score,semantic_similarity,skill_overlap_score,keyword_relevance,qualification_match
0,0,20345168,ACCOUNTANT,1,280,Staff Accountant,Willamette Falls Paper Company,51.66,0.8243,0.1546,0.16,1.0
1,0,20345168,ACCOUNTANT,2,698,Sr. Staff Accountant,Robert Half,50.86,0.8254,0.1134,0.20,1.0
2,0,20345168,ACCOUNTANT,3,1203,Senior Accountant - General Ledger,ParetoHealth,50.09,0.7726,0.1101,0.28,1.0
3,0,20345168,ACCOUNTANT,4,2937,***Accounting Senior Manager | Hybrid in Tampa...,Vaco,49.90,0.7185,0.1759,0.24,1.0
4,0,20345168,ACCOUNTANT,5,958,Accounts Receivable Accountant,Motion Recruitment,49.21,0.7856,0.1233,0.16,1.0
5,1,18569929,ACCOUNTANT,1,280,Staff Accountant,Willamette Falls Paper Company,48.84,0.7536,0.1447,0.16,1.0
6,1,18569929,ACCOUNTANT,2,958,Accounts Receivable Accountant,Motion Recruitment,48.64,0.7622,0.0962,0.24,1.0
7,1,18569929,ACCOUNTANT,3,1203,Senior Accountant - General Ledger,ParetoHealth,48.63,0.7671,0.0909,0.24,1.0
8,1,18569929,ACCOUNTANT,4,2731,Accountant/Staff Accountant,Intellectt Inc,48.23,0.7729,0.0735,0.24,1.0
9,1,18569929,ACCOUNTANT,5,1494,Accounts Receivable Clerk,ASK Consulting,47.36,0.7356,0.1719,0.04,1.0
